## 1 Descripcion del problema
Se buscar analizar los datos del censo para determinar la cantidad de personas por niveles de educación alcanzados en los diferentes departamentos

## 2 Se comenzara cargando los archivos personas y hogares del censo
 luego se cargaran a un data frame pero solo con aquellas columnas que sean relevantes a nuestro trabajo

Primero fui realizando pruebas de carga de archivos completos y luego solo con columnas para detectar posibles errores

In [ ]:
import pandas as pd

df_censo = pd.read_csv("C:/Users/mauri/PycharmProjects/ucu-python-4-data-analysis-project/data/raw/hogares_ext_26_02.csv")

#print(df.head())
df_censo

In [ ]:
df_censo.drop(['Unnamed: 0'], axis=1, inplace=True)
df_censo

In [ ]:
df_filtrado = pd.read_csv("C:/Users/mauri/PycharmProjects/ucu-python-4-data-analysis-project/data/raw/personas_ext_26_02.csv", usecols=["PERED03", "PERED03_1", "PERPH02", "PERED04","DEPARTAMENTO"])
df_filtrado


In [ ]:
df_hog_filtrado = pd.read_csv("C:/Users/mauri/PycharmProjects/ucu-python-4-data-analysis-project/data/raw/hogares_ext_26_02.csv", usecols=["HOGID","DEPARTAMENTO","REGION_4","AREA","MUNICIPIO_PAIS","HOGCE28", "HOGCE11" ,"DIRECCION_ID", "VIVID"])
df_hog_filtrado



### 2.1 se cargan las columnas a utilizar, se unen en un solo archivo, se mapean los nombres de las columnas para que sean entendibles y se guardan en un nuevo archivo csv.

In [1]:
import pandas as pd

# LEER DICCIONARIOS

dic_per = pd.read_excel("C:/Users/mauri/PycharmProjects/ucu-python-4-data-analysis-project/data/raw/Diccionario_de_variables_2023.xlsx",
                        sheet_name="PERSONA", header=None)
dic_hog = pd.read_excel("C:/Users/mauri/PycharmProjects/ucu-python-4-data-analysis-project/data/raw/Diccionario_de_variables_2023.xlsx",
                        sheet_name="HOGAR", header=None)

dic_per.columns = ["desc_var", "nombre_col", "codigo", "desc_codigo", "sub_universo"]
dic_hog.columns = ["desc_var", "nombre_col", "codigo", "desc_codigo", "sub_universo"]

# Extraer mapa código
#def get_mapa(dic, nombre_columna):
 #   filas = dic[dic["nombre_col"] == nombre_columna][["codigo", "desc_codigo"]].dropna()
  #  return dict(zip(filas["codigo"], filas["desc_codigo"]))

def get_mapa(dic, nombre_columna):
    dic_temp = dic.copy()
    dic_temp["nombre_col"] = dic_temp["nombre_col"].ffill()
    filas = dic_temp[dic_temp["nombre_col"] == nombre_columna][["codigo", "desc_codigo"]]
    filas = filas[filas["codigo"].notna() & filas["desc_codigo"].notna()]
    filas = filas[pd.to_numeric(filas["codigo"], errors="coerce").notna()]
    mapa = {}
    for k, v in zip(filas["codigo"], filas["desc_codigo"]):
        mapa[int(k)] = v
    return mapa




# Extraer el nombre
def get_nombre_legible(dic, nombre_columna):
    fila = dic[dic["nombre_col"] == nombre_columna]["desc_var"]
    fila = fila.dropna()
    return fila.iloc[0] if not fila.empty else nombre_columna


# CARGAR archivos csv originales Hogares y personas solo con las columnas a utilizar

cols_per = ["PERED03", "PERED03_1", "PERPH02", "PERED04", "DEPARTAMENTO", "DIRECCION_ID"]
cols_hog = ["REGION_4", "AREA", "MUNICIPIO_PAIS", "HOGCE28", "HOGCE11", "DIRECCION_ID"]

df_per_filtrado = pd.read_csv("C:/Users/mauri/PycharmProjects/ucu-python-4-data-analysis-project/data/raw/personas_ext_26_02.csv", usecols=cols_per)
df_hog_filtrado = pd.read_csv("C:/Users/mauri/PycharmProjects/ucu-python-4-data-analysis-project/data/raw/hogares_ext_26_02.csv",  usecols=cols_hog)


# Unir los archivos
df_final = df_per_filtrado.merge(df_hog_filtrado, on="DIRECCION_ID", how="inner")

# Convertir columnas con códigos a entero antes de mapear
cols_a_convertir = ["PERED03", "PERED03_1", "PERPH02", "PERED04", "REGION_4", "AREA", "HOGCE28", "HOGCE11"]
for col in cols_a_convertir:
    df_final[col] = pd.to_numeric(df_final[col], errors="coerce").astype("Int64")


mapa_especiales = {
    7777: "No aplica",
    8888: "No sabe / Sin dato",
    9898: "No relevado",
    9999: "Sin dato"
}
mapa_departamento = {
    1: "Montevideo", 2: "Artigas", 3: "Canelones", 4: "Cerro Largo",
    5: "Colonia", 6: "Durazno", 7: "Flores", 8: "Florida",
    9: "Lavalleja", 10: "Maldonado", 11: "Paysandú", 12: "Río Negro",
    13: "Rivera", 14: "Rocha", 15: "Salto", 16: "San José",
    17: "Soriano", 18: "Tacuarembó", 19: "Treinta y Tres"

}

mapa_nivel_edu = {
    1: "Educación Inicial o Preescolar",
    2: "Primaria común",
    3: "Primaria especial",
    9: "Magisterio o profesorado",
    10: "Terciario no universitario",
    11: "Universidad o similar",
    12: "Posgrado (diploma, maestría, doctorado)",
    13: "Educación media básica / Ciclo Básico",
    14: "Educación media superior / Bachillerato",
    15: "Capacitaciones UTU (sin CB ni Bach.)",
    **mapa_especiales
}

mapa_finalizo = {1: "Sí", 2: "No", **mapa_especiales}

mapa_computadora = {1: "Sí", 2: "No", **mapa_especiales}

mapa_internet = {1: "Sí", 2: "No", **mapa_especiales}

mapa_sexo = {1: "HOMBRE", 2: "MUJER", **mapa_especiales}

# Aplicar
df_final["DEPARTAMENTO"] = df_final["DEPARTAMENTO"].map(mapa_departamento)
df_final["PERED03"]      = df_final["PERED03"].map(mapa_nivel_edu)
df_final["PERED03_1"]    = df_final["PERED03_1"].map(mapa_nivel_edu)
df_final["PERED04"]      = df_final["PERED04"].map(mapa_finalizo)
df_final["HOGCE28"]      = df_final["HOGCE28"].map(mapa_computadora)
df_final["HOGCE11"]      = df_final["HOGCE11"].map(mapa_internet)
df_final["PERPH02"]  = df_final["PERPH02"].map(mapa_sexo)


# Columnas que vienen de personas
cols_per_renombrar = ["PERED03", "PERED03_1", "PERPH02", "PERED04", "DEPARTAMENTO"]
# Columnas que vienen de hogares
cols_hog_renombrar = ["REGION_4", "AREA", "MUNICIPIO_PAIS", "HOGCE28", "HOGCE11"]

renombrar = {}
for col in cols_per_renombrar:
    renombrar[col] = get_nombre_legible(dic_per, col)
for col in cols_hog_renombrar:
    renombrar[col] = get_nombre_legible(dic_hog, col)

df_final = df_final.rename(columns=renombrar)

df_final.to_csv("C:/Users/mauri/PycharmProjects/ucu-python-4-data-analysis-project/data/processed/censo_procesado_todas_las_personas.csv", index=False)

df_final


,DIRECCION_ID,DEPARTAMENTO,SEXO AL NACER,NIVEL EDUCATIVO CURSANDO ACTUALMENTE,NIVEL MÁS ALTO QUE CURSÓ,FINALIZÓ ESE NIVEL,REGIÓN,AREA,MUNICIPIO,"COMPUTADORA, LAPTOP, NOTEBOOK O TABLET",ACCESO A INTERNET
0,1146859.0,Montevideo,MUJER,No sabe / Sin dato,No sabe / Sin dato,No sabe / Sin dato,1,1,Municipio E,No sabe / Sin dato,No sabe / Sin dato
1,1146859.0,Montevideo,HOMBRE,No sabe / Sin dato,No sabe / Sin dato,No sabe / Sin dato,1,1,Municipio E,No sabe / Sin dato,No sabe / Sin dato
2,1313925.0,Montevideo,MUJER,No sabe / Sin dato,No sabe / Sin dato,No sabe / Sin dato,1,1,Municipio D,No sabe / Sin dato,No sabe / Sin dato
3,1313925.0,Montevideo,MUJER,No sabe / Sin dato,No sabe / Sin dato,No sabe / Sin dato,1,1,Municipio D,No sabe / Sin dato,No sabe / Sin dato
4,1313925.0,Montevideo,MUJER,No sabe / Sin dato,No sabe / Sin dato,No sabe / Sin dato,1,1,Municipio D,No sabe / Sin dato,No sabe / Sin dato
...,...,...,...,...,...,...,...,...,...,...,...
3217527,16053338.0,San José,HOMBRE,No sabe / Sin dato,No aplica,No aplica,3,1,NaN,No aplica,No aplica
3217528,16053338.0,San José,HOMBRE,No sabe / Sin dato,Primaria común,No aplica,3,1,NaN,No aplica,No aplica
3217529,16053338.0,San José,MUJER,No sabe / Sin dato,Primaria común,No aplica,3,1,NaN,No aplica,No aplica
3217530,16053338.0,San José,MUJER,No sabe / Sin dato,Primaria común,No aplica,3,1,NaN,No aplica,No aplica
